# Где IT-специалисту в ЕС выгоднее жить и работать? (2025/2026)

## Прикладной проект на реальных данных (страны Евросоюза)

Цель — найти страны ЕС, где IT-специалист (роль **Software Developer**) получает **больше денег на руки** после подоходного налога и социальных взносов, а также аренды жилья, то есть максимизирует **располагаемый доход**, при более низкой стоимости жилья и налогах.

### Особенности ЕС (важно):
- Налоги — **на уровне страны** и сильно различаются (как штаты в США): подоходный налог **+ социальные взносы работника** от ~21% (Эстония) до ~42% (Бельгия).
- Разные **валюты**: PLN, CZK, SEK, DKK, HUF, RON приведены к **EUR** по курсам 2025.
- Данные — оценки **2025 года** (актуально для 2025/2026).

### Источники данных (реальные):
| Показатель | Источник | Период |
|---|---|---|
| Зарплата Software Developer | **Stack Overflow Survey / Glassdoor / Eurostat** | 2025 |
| Аренда (1-комн., центр) | **Numbeo / Eurostat** | 2025 |
| Цена жилья за м² (центр) | **Numbeo / Eurostat HPI** | 2025 |
| Подоходный налог + соц. взносы | **OECD Taxing Wages / PwC** | 2024–2025 |

> Единого бесплатного API уровня Census для IT-зарплат и аренды в ЕС нет, поэтому используется **встроенный снимок** опубликованных значений (см. «Источники и оговорки»).

### Содержание:
1. Загрузка данных
2. Exploratory Data Analysis (EDA)
3. Налог + соц. взносы и доход «на руки»
4. Жильё: аренда vs ипотека (ставка еврозоны 2025)
5. Располагаемый доход и доступность жилья
6. Композитный рейтинг стран
7. Анализ чувствительности
8. Выводы и рекомендации

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')
pd.set_option('display.width', 180)
pd.set_option('display.max_columns', 30)

print('Библиотеки успешно загружены!')

## 1. Загрузка данных

Встроенный снимок реальных опубликованных значений (2025) по 21 стране ЕС.

### Столбцы:
- **gross** — средняя годовая зарплата Software Developer (gross), €
- **rent** — аренда 1-комн. квартиры в центре столицы, €/мес
- **price_m2** — цена жилья за м² в центре, €
- **tax_social** — эффективная ставка (подоходный налог + соц. взносы работника)

In [ ]:
# Встроенный снимок 2025: страна -> (код, gross_€/год, rent_€/мес, price_m2_€, tax_social)
#  gross:      Stack Overflow Survey / Glassdoor / Eurostat, средняя годовая (gross), €
#  rent:       Numbeo, аренда 1-комн. в центре столицы, €/мес
#  price_m2:   Numbeo, цена жилья за м² в центре, €
#  tax_social: OECD Taxing Wages / PwC — эфф. ставка (НДФЛ + соц.взносы работника),
#              приблизительно для одинокого работника на указанной зарплате
SNAPSHOT = {
  'Люксембург':    ('LU', 85000, 1800, 11000, 0.29),
  'Дания':         ('DK', 75000, 1500,  6000, 0.36),
  'Ирландия':      ('IE', 70000, 2100,  6500, 0.28),
  'Германия':      ('DE', 68000, 1250,  7000, 0.39),
  'Нидерланды':    ('NL', 66000, 1900,  7500, 0.32),
  'Австрия':       ('AT', 62000, 1050,  8000, 0.34),
  'Швеция':        ('SE', 60000, 1400,  7000, 0.30),
  'Финляндия':     ('FI', 58000, 1150,  5500, 0.33),
  'Бельгия':       ('BE', 58000, 1050,  4000, 0.42),
  'Франция':       ('FR', 52000, 1300, 10000, 0.29),
  'Испания':       ('ES', 42000, 1200,  4500, 0.24),
  'Эстония':       ('EE', 42000,  800,  3500, 0.21),
  'Италия':        ('IT', 40000, 1200,  5000, 0.33),
  'Польша':        ('PL', 40000,  950,  4000, 0.24),
  'Литва':         ('LT', 40000,  800,  3000, 0.29),
  'Чехия':         ('CZ', 38000, 1000,  5500, 0.23),
  'Румыния':       ('RO', 36000,  600,  2200, 0.40),
  'Португалия':    ('PT', 35000, 1300,  5000, 0.29),
  'Венгрия':       ('HU', 32000,  650,  3000, 0.33),
  'Греция':        ('GR', 28000,  600,  2800, 0.30),
  'Хорватия':      ('HR', 30000,  700,  3200, 0.27),
}

df = pd.DataFrame(
    [(c, v[0], v[1], v[2], v[3], v[4]) for c, v in SNAPSHOT.items()],
    columns=['country', 'code', 'gross', 'rent', 'price_m2', 'tax_social']
)
DATA_SOURCE = 'встроенный снимок 2025 (Stack Overflow / Glassdoor / Numbeo / OECD)'
print(f'Загружено стран: {len(df)}')
print(f'Источник: {DATA_SOURCE}')
print(df.head())

## 2. Exploratory Data Analysis (EDA)

Изучим распределение зарплат, аренды, цены жилья и налоговой нагрузки по странам ЕС.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Зарплата разработчика по странам
top_wage = df.sort_values('gross')
axes[0, 0].barh(top_wage['country'], top_wage['gross'] / 1000, color='steelblue', alpha=0.8)
axes[0, 0].set_xlabel('Зарплата (тыс. €/год, gross)')
axes[0, 0].set_title('Зарплата Software Developer по странам ЕС (2025)')

# 2. Зарплата vs аренда
axes[0, 1].scatter(df['rent'], df['gross'] / 1000, s=70, alpha=0.7, color='coral')
for _, r in df.iterrows():
    axes[0, 1].annotate(r['code'], (r['rent'], r['gross'] / 1000), fontsize=8)
axes[0, 1].set_xlabel('Аренда (€/мес)')
axes[0, 1].set_ylabel('Зарплата (тыс. €/год)')
axes[0, 1].set_title('Зарплата vs аренда')

# 3. Налоговая нагрузка
top_tax = df.sort_values('tax_social')
colors = plt.cm.RdYlGn_r(top_tax['tax_social'] / top_tax['tax_social'].max())
axes[1, 0].barh(top_tax['country'], top_tax['tax_social'] * 100, color=colors)
axes[1, 0].set_xlabel('Налог + соц. взносы (%)')
axes[1, 0].set_title('Налоговая нагрузка работника')

# 4. Цена жилья за м²
top_price = df.sort_values('price_m2')
axes[1, 1].barh(top_price['country'], top_price['price_m2'] / 1000, color='green', alpha=0.7)
axes[1, 1].set_xlabel('Цена жилья (тыс. €/м²)')
axes[1, 1].set_title('Стоимость жилья в центре столицы')

plt.tight_layout()
plt.show()

print('Ключевые статистики:')
print('=' * 60)
print(f'Медианная зарплата: €{df["gross"].median():,.0f}/год')
print(f'Разброс: €{df["gross"].min():,.0f} — €{df["gross"].max():,.0f}')
print(f'Налоговая нагрузка: {df["tax_social"].min()*100:.0f}% — {df["tax_social"].max()*100:.0f}%')
print(f'Медианная аренда: €{df["rent"].median():,.0f}/мес')

## 3. Налог + социальные взносы и доход «на руки»

В ЕС с работника удерживаются **подоходный налог** и **социальные взносы**. Используем эффективную совокупную ставку для расчёта дохода на руки:

$$\text{net} = \text{gross} \times (1 - \text{tax\_social})$$

Ставки различаются по странам — это ключевой фактор «где меньше налоги».

In [ ]:
df['net_year'] = df['gross'] * (1 - df['tax_social'])
df['net_month'] = df['net_year'] / 12

print('Доход «на руки» после налога и соц. взносов')
print('=' * 70)
show = df.sort_values('net_year', ascending=False)[
    ['country', 'gross', 'tax_social', 'net_year', 'net_month']].head(12).copy()
show['gross'] = show['gross'].map('€{:,.0f}'.format)
show['tax_social'] = (show['tax_social'] * 100).map('{:.0f}%'.format)
show['net_year'] = show['net_year'].map('€{:,.0f}'.format)
show['net_month'] = show['net_month'].map('€{:,.0f}'.format)
print(show.to_string(index=False))

# gross vs net для топ-12 по зарплате
top = df.nlargest(12, 'gross').sort_values('gross')
fig, ax = plt.subplots(figsize=(12, 6))
y = np.arange(len(top))
ax.barh(y, top['gross'] / 1000, color='lightgray', label='Gross')
ax.barh(y, top['net_year'] / 1000, color='steelblue', alpha=0.9, label='На руки (net)')
ax.set_yticks(y)
ax.set_yticklabels(top['country'])
ax.set_xlabel('тыс. €/год')
ax.set_title('Зарплата до и после налогов (топ-12 по gross)')
ax.legend()
for i, r in enumerate(top.itertuples()):
    ax.text(r.net_year / 1000 - 1, i, f'{r.net_year/1000:.0f}',
            va='center', ha='right', color='white', fontsize=8, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Жильё: аренда vs ипотека

Годовые расходы на жильё:
- **Аренда:** `rent * 12`.
- **Покупка:** квартира 60 м², первоначальный взнос 20%, ставка **3,5%** (средняя по еврозоне в 2025), срок 30 лет.

Месячный платёж: $M = P\dfrac{r(1+r)^n}{(1+r)^n-1}$.

In [ ]:
EU_MORTGAGE_RATE = 0.035   # средняя ставка еврозоны 2025
DOWN = 0.20
YEARS = 30
AREA = 60  # м²

def annual_mortgage(price_m2):
    P = price_m2 * AREA * (1 - DOWN)
    r = EU_MORTGAGE_RATE / 12
    n = YEARS * 12
    M = P * r * (1 + r) ** n / ((1 + r) ** n - 1)
    return M * 12

df['home_price'] = df['price_m2'] * AREA
df['annual_rent'] = df['rent'] * 12
df['annual_mortgage'] = df['price_m2'].apply(annual_mortgage)

df['dispo_rent'] = df['net_year'] - df['annual_rent']
df['dispo_buy'] = df['net_year'] - df['annual_mortgage']
df['rent_burden'] = df['annual_rent'] / df['net_year']
df['price_to_income'] = df['home_price'] / df['gross']  # лет gross-зарплаты на квартиру 60м²

print('Располагаемый доход после аренды — топ-10')
print('=' * 80)
cols = ['country', 'net_year', 'annual_rent', 'dispo_rent', 'rent_burden', 'price_to_income']
t = df.sort_values('dispo_rent', ascending=False)[cols].head(10).copy()
for c in ['net_year', 'annual_rent', 'dispo_rent']:
    t[c] = t[c].map('€{:,.0f}'.format)
t['rent_burden'] = (t['rent_burden'] * 100).map('{:.0f}%'.format)
t['price_to_income'] = t['price_to_income'].map('{:.1f} лет'.format)
print(t.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

top_dispo = df.nlargest(15, 'dispo_rent').sort_values('dispo_rent')
axes[0].barh(top_dispo['country'], top_dispo['dispo_rent'] / 1000, color='seagreen', alpha=0.8)
axes[0].set_xlabel('Располагаемый доход после аренды (тыс. €/год)')
axes[0].set_title('Топ-15 стран: деньги на руки после налогов и аренды')

sc = axes[1].scatter(df['annual_rent'] / 1000, df['net_year'] / 1000,
                     s=(df['dispo_rent'] / df['dispo_rent'].max() * 400).clip(30),
                     c=df['dispo_rent'] / 1000, cmap='RdYlGn', alpha=0.8, edgecolor='k')
for _, r in df.iterrows():
    axes[1].annotate(r['code'], (r['annual_rent'] / 1000, r['net_year'] / 1000), fontsize=8)
axes[1].set_xlabel('Годовая аренда (тыс. €)')
axes[1].set_ylabel('Доход на руки (тыс. €/год)')
axes[1].set_title('Аренда vs доход (цвет/размер = располагаемый доход)')
plt.colorbar(sc, ax=axes[1], label='Располагаемый доход (тыс. €)')

plt.tight_layout()
plt.show()

## 5–6. Композитный рейтинг стран

Стандартизируем показатели через **z-оценку** и построим взвешенную сумму:

$$\text{score} = \sum_i w_i \cdot z_i, \qquad z_i = \frac{x_i - \bar{x}_i}{\sigma_i}$$

Знак веса отрицательный для «чем меньше — тем лучше» (аренда, налог, цена жилья).

In [ ]:
def zscore(s):
    return (s - s.mean()) / s.std(ddof=0)

WEIGHTS = {
    'net_year':        0.35,   # доход на руки
    'dispo_rent':      0.30,   # что остаётся после аренды
    'annual_rent':    -0.15,   # аренда — меньше лучше
    'tax_social':     -0.10,   # налог — меньше лучше
    'price_to_income':-0.10,   # доступность покупки — меньше лучше
}

def compute_score(data, weights):
    score = pd.Series(0.0, index=data.index)
    for col, w in weights.items():
        score += w * zscore(data[col])
    return score

df['score'] = compute_score(df, WEIGHTS)
df['rank'] = df['score'].rank(ascending=False).astype(int)
ranking = df.sort_values('score', ascending=False).reset_index(drop=True)

print('ИТОГОВЫЙ РЕЙТИНГ СТРАН ЕС ДЛЯ IT-СПЕЦИАЛИСТА (2025/2026)')
print('=' * 85)
view = ranking[['rank', 'country', 'gross', 'net_year', 'tax_social',
                'annual_rent', 'dispo_rent', 'score']].head(15).copy()
for c in ['gross', 'net_year', 'annual_rent', 'dispo_rent']:
    view[c] = view[c].map('€{:,.0f}'.format)
view['tax_social'] = (view['tax_social'] * 100).map('{:.0f}%'.format)
view['score'] = view['score'].map('{:+.2f}'.format)
print(view.to_string(index=False))

In [ ]:
# Тепловая карта нормированных метрик для топ-12
top12 = ranking.head(12).set_index('country')
metrics = ['net_year', 'dispo_rent', 'annual_rent', 'tax_social', 'price_to_income']
labels = ['Доход на руки', 'Располаг. доход', 'Аренда/год', 'Налог+взносы', 'Цена/доход']

norm = pd.DataFrame({m: zscore(df.set_index('country')[m]) for m in metrics})
norm = norm.loc[top12.index]
for m in ['annual_rent', 'tax_social', 'price_to_income']:
    norm[m] = -norm[m]  # зелёный = хорошо
norm.columns = labels

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(norm, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            linewidths=0.5, cbar_kws={'label': 'z-оценка (зелёный = лучше)'}, ax=ax)
ax.set_title('Профиль топ-12 стран по нормированным метрикам')
ax.set_ylabel('')
plt.tight_layout()
plt.show()

## 7. Анализ чувствительности к весам

Сравним приоритеты: «максимум дохода», «минимум налогов и расходов», «сбалансированный».

In [ ]:
scenarios = {
    'Максимум дохода': {'net_year': 0.55, 'dispo_rent': 0.35,
                        'annual_rent': -0.05, 'tax_social': -0.05, 'price_to_income': 0.0},
    'Минимум расходов': {'net_year': 0.10, 'dispo_rent': 0.20,
                         'annual_rent': -0.30, 'tax_social': -0.20, 'price_to_income': -0.20},
    'Сбалансированный': WEIGHTS,
}

result = df[['country', 'code']].copy()
for name, w in scenarios.items():
    result[name] = compute_score(df, w).rank(ascending=False).astype(int)

top_c = df.nlargest(10, 'score')['code'].tolist()
comp = result[result['code'].isin(top_c)].sort_values('Сбалансированный')
print('Ранги стран в разных сценариях (топ-10 сбалансированного)')
print('=' * 70)
print(comp.to_string(index=False))

fig, ax = plt.subplots(figsize=(11, 7))
sc_names = list(scenarios.keys())
for _, row in comp.iterrows():
    ranks = [row[s] for s in sc_names]
    ax.plot(sc_names, ranks, marker='o', label=row['country'])
ax.invert_yaxis()
ax.set_ylabel('Ранг (1 = лучший)')
ax.set_title('Как меняется рейтинг страны в зависимости от приоритетов')
ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## 8. Выводы и рекомендации

In [ ]:
best = ranking.iloc[0]
top5 = ranking.head(5)

print('РЕКОМЕНДАЦИИ ДЛЯ IT-СПЕЦИАЛИСТА В ЕС (Software Developer, 2025/2026)')
print('=' * 70)
print(f'Источник данных: {DATA_SOURCE}')
print('-' * 70)
print(f'Лучшая страна по совокупности критериев: {best["country"]}')
print(f'  Зарплата (gross):    €{best["gross"]:,.0f}/год')
print(f'  На руки (после налогов): €{best["net_year"]:,.0f}/год (€{best["net_month"]:,.0f}/мес)')
print(f'  Налог + взносы:      {best["tax_social"]*100:.0f}%')
print(f'  Годовая аренда:      €{best["annual_rent"]:,.0f}')
print(f'  Располагаемый доход: €{best["dispo_rent"]:,.0f}/год')
print('-' * 70)
print('ТОП-5 СТРАН:')
for _, r in top5.iterrows():
    print(f'  {int(r["rank"])}. {r["country"]:<14} располаг. доход €{r["dispo_rent"]:,.0f} | '
          f'налог {r["tax_social"]*100:.0f}% | аренда €{r["rent"]:,.0f}/мес')
print('-' * 70)
print('НАБЛЮДЕНИЯ:')
print(f'  - Самая высокая зарплата: {df.loc[df["gross"].idxmax(), "country"]} '
      f'(€{df["gross"].max():,.0f}).')
print(f'  - Самые низкие налоги: {df.loc[df["tax_social"].idxmin(), "country"]} '
      f'({df["tax_social"].min()*100:.0f}%).')
print(f'  - Самая доступная аренда: {df.loc[df["rent"].idxmin(), "country"]} '
      f'(€{df["rent"].min():,.0f}/мес).')
print('  - Высокая зарплата не гарантирует лучший располагаемый доход:')
print('    высокие налоги (BE/DE) и аренда (IE/NL/LU) снижают выгоду.')

## Источники и оговорки

**Источники данных:**
- Зарплаты: **Stack Overflow Developer Survey 2025**, **Glassdoor**, **Eurostat** (structure of earnings). Значения — средние по стране, gross.
- Аренда и цена жилья: **Numbeo** (2025), **Eurostat** House Price Index — [ec.europa.eu/eurostat](https://ec.europa.eu/eurostat) (есть открытый API).
- Налоги и взносы: **OECD Taxing Wages 2024/2025**, **PwC Worldwide Tax Summaries**.

**Оговорки (важно для корректной интерпретации):**
1. `tax_social` — **эффективная** совокупная ставка (подоходный налог + соц. взносы работника) для одинокого работника; реальная зависит от вычетов, региона, семьи, спец. режимов (напр. IP Box в Польше, 30%-ruling в Нидерландах, impatriate-режимы).
2. Суммы в **EUR**; для не-евро стран (PLN, CZK, SEK, DKK, HUF, RON) применён курс 2025 — он влияет на сравнение.
3. Не учтена **покупательная способность (PPP)**: €50k в Португалии ≠ €50k в Люксембурге (см. упражнения).
4. Аренда/цена — по **центру столицы**; в регионах дешевле.
5. Не учтены: медстраховка (если не в соц. взносах), НДС, налог на недвижимость, коммунальные платежи.
6. Зарплаты зависят от грейда (junior/middle/senior) и стека — здесь усреднённые.

## Упражнения

### Упражнение 1: Покупательная способность (PPP)
1. Добавьте индекс цен Eurostat (PLI) и пересчитайте доход в PPP
2. Как меняется рейтинг? (Португалия/Польша обычно поднимаются)

### Упражнение 2: Спец. налоговые режимы
1. Учтите нидерландский 30%-ruling и польский IP Box (5%)
2. Насколько это меняет привлекательность NL и PL?

### Упражнение 3: Аренда vs покупка
1. Постройте рейтинг по `dispo_buy` (ипотека) вместо аренды
2. При какой ставке покупка выгоднее аренды в топ-странах?

### Упражнение 4: Свежие данные через Eurostat API
1. Загрузите House Price Index и average earnings из открытого API Eurostat
2. Сравните с встроенным снимком

### Упражнение 5: Свои приоритеты
1. Задайте свои веса в `WEIGHTS` (например, налоги важнее всего)
2. Как меняется ваш личный топ-5?

---

**Решения** можно найти в ноутбуке `solutions/20_Solutions.ipynb`